# 8-тарау. GAN: жалған жасаушы мен сарапшы

*«Машина қалай үйренеді? 2-том» кітабына арналған блокнот. Әр ұяшықтың түсіндірмесі кітапта берілген.*

Ұяшықтарды жоғарыдан төмен қарай ретімен іске қосыңыз: **Shift + Enter**.

## 8.1. Жалған ақша жасаушы мен банк сарапшысы

## 8.2. Екі желі, екі мақсат

## 8.3. Екі өлшемді жаттығу

In [ ]:
import torch
from torch import nn

torch.manual_seed(0)
def real_ring(n):
    angle = torch.rand(n) * 2 * torch.pi
    pts = torch.stack([torch.cos(angle), torch.sin(angle)], dim=1)
    return pts + 0.05 * torch.randn(n, 2)

G = nn.Sequential(nn.Linear(2, 64), nn.ReLU(),
                  nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 2))
D = nn.Sequential(nn.Linear(2, 64), nn.LeakyReLU(0.2),
                  nn.Linear(64, 64), nn.LeakyReLU(0.2), nn.Linear(64, 1))
opt_G = torch.optim.Adam(G.parameters(), lr=1e-3, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=1e-3, betas=(0.5, 0.999))
bce = nn.BCEWithLogitsLoss()

In [ ]:
ones, zeros = torch.ones(256, 1), torch.zeros(256, 1)
z_fixed = torch.randn(500, 2)        # бақылауға арналған тұрақты шу
snapshots = {}
for step in range(4001):
    real = real_ring(256)
    fake = G(torch.randn(256, 2))
    # 1) дискриминатор: шынды 1, жалғанды 0 деп тануды үйренеді
    loss_D = bce(D(real), ones) + bce(D(fake.detach()), zeros)
    opt_D.zero_grad()
    loss_D.backward()
    opt_D.step()
    # 2) генератор: дискриминатор «1» деп айтуына тырысады
    loss_G = bce(D(fake), ones)
    opt_G.zero_grad()
    loss_G.backward()
    opt_G.step()
    if step in (0, 300, 1000, 2000, 4000):
        with torch.no_grad():
            p = G(z_fixed)
        snapshots[step] = p
        r = p.norm(dim=1)
        sectors = torch.histc(torch.atan2(p[:, 1], p[:, 0]), bins=8,
                              min=-torch.pi, max=torch.pi)
        print(f"{step:4d}: радиус={r.mean():.2f}±{r.std():.2f}, "
              f"8 сектордың {int((sectors >= 20).sum())}-і толды")

## 8.4. Цифрлар жасаймыз

In [ ]:
import urllib.request
import numpy as np
from torch.utils.data import TensorDataset, DataLoader

URL = ("https://storage.googleapis.com/tensorflow/"
       "tf-keras-datasets/mnist.npz")
urllib.request.urlretrieve(URL, "mnist.npz")
d = np.load("mnist.npz")
X = torch.tensor(d["x_train"], dtype=torch.float32).view(-1, 784)
X = X / 255 * 2 - 1                         # [-1; 1] аралығы
dl = DataLoader(TensorDataset(X), batch_size=128,
                shuffle=True, drop_last=True)

torch.manual_seed(0)
G = nn.Sequential(nn.Linear(64, 256), nn.LeakyReLU(0.2),
                  nn.Linear(256, 512), nn.LeakyReLU(0.2),
                  nn.Linear(512, 784), nn.Tanh())
D = nn.Sequential(nn.Linear(784, 512), nn.LeakyReLU(0.2),
                  nn.Linear(512, 256), nn.LeakyReLU(0.2),
                  nn.Linear(256, 1))
opt_G = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

In [ ]:
z_fixed = torch.randn(32, 64)
progress = {}
for epoch in range(1, 31):
    for (xb,) in dl:
        n = len(xb)
        fake = G(torch.randn(n, 64))
        loss_D = (bce(D(xb), torch.ones(n, 1)) +
                  bce(D(fake.detach()), torch.zeros(n, 1)))
        opt_D.zero_grad()
        loss_D.backward()
        opt_D.step()
        loss_G = bce(D(fake), torch.ones(n, 1))
        opt_G.zero_grad()
        loss_G.backward()
        opt_G.step()
    if epoch in (1, 5, 30):
        with torch.no_grad():
            progress[epoch] = (G(z_fixed) + 1) / 2   # [0; 1] аралығына
    if epoch % 10 == 0:
        print(f"{epoch:2d}-дәуір: D={loss_D.item():.3f}, "
              f"G={loss_G.item():.3f}")

## 8.5. GAN-ды оқытудың қиындықтары